In [1]:
import pandas as pd
import os 
import glob

In [ ]:
program_path = os.path.abspath('./')
data_path = os.path.join(program_path, 'data') #Esta línea es compatible tanto para MacOS como Windows path.join utiliza las convenciones del sistema operativo en el que se utiliza.
data_path


'c:\\Users\\super\\Documents\\Maestria\\MLOPS\\MiBici\\data'

In [3]:
csv_data_list = glob.glob('\\'.join([data_path, '*datos*.csv']))
display(csv_data_list)

# La nomenclatura no tiene ningún carácter especial, por lo tanto puede ser leída utilizando el encoding por default UTF-8
station_data = pd.read_csv(f'{data_path}/nomenclatura_2026_09.csv')

['c:\\Users\\super\\Documents\\Maestria\\MLOPS\\MiBici\\data\\datos_abiertos_2026_01.csv']

In [4]:
data_df_list = [pd.read_csv(d, encoding='latin-1') for d in csv_data_list] #Crear una lista de dataframes. Al final esta lista se combinará utilizando merge

#Todos los dfs de datos tienen la misma estructura entonces podemos concatenarlos con la función concat
concat_data_df = pd.concat(data_df_list, ignore_index=True) #Reiniciar indice después de concatenar

In [5]:
#Pequeña comprobación para ver si hay viajes duplicados
print(f"Viajes duplicados: {concat_data_df['Viaje_Id'].duplicated().sum():,}")

Viajes duplicados: 0


In [6]:
#Añadir la información de la nomenclatura - estacion de origen
concat_data_df = concat_data_df.merge(
    station_data[["id", "name","latitude", "longitude", "dpcapacity"]], 
    left_on="Origen_Id", 
    right_on="id",
    how='left'
).drop(columns = "id") #Deshacerse de la columna id del DF de nomenclatura despues de concatenar

#Renombrar las columnas para diferenciar el origen de la salida
concat_data_df = concat_data_df.rename(columns={
    'name': 'Origen_nombre',
    'latitude': 'Origen_latitud',
    'longitude': 'Origen_longitud',
    'dpcapacity': 'Origen_capacidad'
})

# Información de la estación de destino
concat_data_df = concat_data_df.merge(
    station_data[['id', 'name', 'latitude', 'longitude', 'dpcapacity']],
    left_on='Destino_Id',
    right_on='id',
    how='left'
).drop(columns='id')


#Renombrar las columnas para diferenciar el destino del origen
concat_data_df = concat_data_df.rename(columns={
    'name': 'Destino_nombre',
    'latitude': 'Destino_latitud',
    'longitude': 'Destino_longitud',
    'dpcapacity': 'Destino_capacidad'
})

In [7]:
concat_data_df.to_csv(
    './MiBici_datos_base.csv',
    index=False,
    encoding='utf-8'
)